# MemSolve ImageNet training

Use an existing current-source checkout and its matching CUDA build. This notebook
checks the environment and dataset, launches one detached T/S/B job, and monitors it.
It does not download an old release wheel or replace the installed PyTorch stack.
All tiers use RoPE-ViT-derived 400-epoch pretraining with global batch 2048.
T/S use 224px; Base uses 192px then a separate 20-epoch, 224px fine-tuning phase with batch 512.
BF16 AMP, persistent workers and compile-graph are retained. EMA decay is 0.99996; ordinary and EMA validation metrics and best checkpoints are tracked separately.
See `docs/IMAGENET_DEIT3.md` and `docs/IMAGENET_VIT3.md` for provenance and adaptations.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
import sys
import time
from pathlib import Path

REPOSITORY_DIR = Path(os.environ.get('MEMSOLVE_REPOSITORY_DIR', str(next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'memsolve').is_dir() and (p / 'pyproject.toml').is_file()), Path.home() / 'MemSolve')))).expanduser()
TARGET_PYTHON = Path(os.environ.get('MEMSOLVE_PYTHON', sys.executable)).expanduser()
DATA_ROOT = Path(os.environ.get('MEMSOLVE_IMAGENET_WDS_ROOT', str(Path.home() / 'data' / 'imagenet-1k-wds'))).expanduser()
RUN_ROOT = Path(os.environ.get('MEMSOLVE_IMAGENET_RUN_ROOT', str(Path.home() / 'runs' / 'memsolve-imagenet'))).expanduser()
CONTROL_ROOT = Path(os.environ.get('MEMSOLVE_IMAGENET_CONTROL_ROOT', str(Path.home() / '.cache' / 'memsolve-imagenet'))).expanduser()
TIER = 'small'  # tiny, small, base
if TIER not in {'tiny', 'small', 'base'}:
    raise ValueError('choose tiny, small, or base')
PHASE = 'pretrain'  # Base also supports 'finetune'; run it after pretraining.
FINETUNE_CHECKPOINT = None  # Path to the compatible Base pretraining checkpoint.
if PHASE not in {'pretrain', 'finetune'} or (PHASE == 'finetune' and TIER != 'base'):
    raise ValueError('choose pretrain, or finetune with tier base')
if PHASE == 'finetune' and FINETUNE_CHECKPOINT is None:
    raise ValueError('set FINETUNE_CHECKPOINT for Base resolution fine-tuning')
PHASE_ARGS = ['--phase', PHASE]
if PHASE == 'finetune':
    PHASE_ARGS += ['--finetune', str(Path(FINETUNE_CHECKPOINT).expanduser())]
CONFIG_NAME = 'imagenet_ropevit_400.toml'
CONFIG_PATH = REPOSITORY_DIR / 'experiments' / 'configs' / CONFIG_NAME
RUN_NAME = f'{CONFIG_PATH.stem}-{TIER}-{PHASE}'
PHYSICAL_BATCH_SIZE = None  # Resolve 512 for pretraining, 256 for fine-tuning.
TRAIN_WORKERS = None
VAL_WORKERS = None
NPROC_PER_NODE = None  # None uses all visible GPUs.
CUDA_VISIBLE_DEVICES = None
TARGET_ENV = os.environ.copy()
TARGET_ENV['PATH'] = str(TARGET_PYTHON.parent) + os.pathsep + TARGET_ENV.get('PATH', '')
TARGET_ENV['PYTHONPATH'] = str(REPOSITORY_DIR) + os.pathsep + TARGET_ENV.get('PYTHONPATH', '')
if CUDA_VISIBLE_DEVICES is not None:
    TARGET_ENV['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES
for directory in (RUN_ROOT, CONTROL_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

def show_command(command):
    print('$ ' + ' '.join(shlex.quote(part) for part in command))

def run(command, *, cwd=None, env=None, capture=False):
    show_command(command)
    return subprocess.run(command, cwd=cwd, env=env, check=True, text=True, capture_output=capture)

def output(command, *, env=None):
    return run(command, env=env, capture=True).stdout.strip()


In [ ]:
from __future__ import annotations

if not CONFIG_PATH.is_file():
    raise FileNotFoundError(CONFIG_PATH)
if not TARGET_PYTHON.is_file():
    raise FileNotFoundError(TARGET_PYTHON)
probe = "import torch, timm, torchvision, webdataset; print(torch.__version__, torch.version.cuda); assert torch.cuda.is_available()"
if PHASE == 'pretrain':
    probe += '; import wids; from apex.optimizers import FusedLAMB; FusedLAMB([torch.nn.Parameter(torch.ones(1, device="cuda"))])'
run([str(TARGET_PYTHON), '-c', probe], cwd=REPOSITORY_DIR, env=TARGET_ENV)


In [ ]:
from __future__ import annotations

verification = """
import torch
from memsolve import MemSolve, MemSolveConfig
from memsolve.ball import cuda
cuda.load(device=torch.device('cuda'))
layer = MemSolve(MemSolveConfig(dim=64, num_heads=1, rank=16)).cuda()
x = torch.randn(2, 17, 64, device='cuda', dtype=torch.bfloat16, requires_grad=True)
y = layer(x, implementation='cuda')
y.float().square().mean().backward()
assert torch.isfinite(y).all() and torch.isfinite(x.grad).all()
print('current native runtime forward/backward passed')
"""
run([str(TARGET_PYTHON), '-c', verification], cwd=REPOSITORY_DIR, env=TARGET_ENV)


In [ ]:
from __future__ import annotations

def validate_webdataset_root(root: Path) -> tuple[dict[str, object] | None, str]:
    # Execute through TARGET_PYTHON so this notebook and the trainer share one manifest contract.
    validation = '''
import json
import sys
from pathlib import Path
from experiments.imagenet import load_imagenet_webdataset_manifest

manifest = load_imagenet_webdataset_manifest(Path(sys.argv[1]))
print(json.dumps({
    'manifest_sha256': manifest.sha256,
    'train_shards': len(manifest.train.paths),
    'validation_shards': len(manifest.validation.paths),
    'train_samples': manifest.train.num_samples,
    'validation_samples': manifest.validation.num_samples,
    'first_train_shard': manifest.train.paths[0].name,
    'last_train_shard': manifest.train.paths[-1].name,
    'first_validation_shard': manifest.validation.paths[0].name,
    'last_validation_shard': manifest.validation.paths[-1].name,
}, sort_keys=True))
'''
    completed = subprocess.run(
        [str(TARGET_PYTHON), '-c', validation, str(root)],
        cwd=REPOSITORY_DIR,
        env=TARGET_ENV,
        text=True,
        capture_output=True,
    )
    if completed.returncode != 0:
        detail = (completed.stderr or completed.stdout).strip()
        return None, detail[-2000:] or 'the ImageNet WebDataset validator failed'
    try:
        return json.loads(completed.stdout), 'verified'
    except json.JSONDecodeError:
        return None, f'validator produced invalid JSON: {completed.stdout[-1000:]}'

ready, status = validate_webdataset_root(DATA_ROOT)
if ready is None:
    raise FileNotFoundError(
        'ImageNet-1K WebDataset is not ready. Set MEMSOLVE_IMAGENET_WDS_ROOT to a directory containing '
        'the prepared timm/imagenet-1k-wds tar shards. '
        f'Validator: {status}'
    )
print({'webdataset_root': str(DATA_ROOT), **ready})


In [ ]:
from __future__ import annotations

def visible_gpu_count() -> int:
    return int(output([str(TARGET_PYTHON), '-c', 'import torch; print(torch.cuda.device_count())'], env=TARGET_ENV))

world_size = NPROC_PER_NODE or visible_gpu_count()
recipe_probe = "import json, sys; from experiments.imagenet import load_run, parse_args; print(json.dumps(load_run(parse_args(sys.argv[1:])).train))"
recipe = json.loads(output([str(TARGET_PYTHON), '-c', recipe_probe, '--config', str(CONFIG_PATH), '--tier', TIER, '--data-root', str(DATA_ROOT), '--output', str(RUN_ROOT / RUN_NAME), *PHASE_ARGS], env=TARGET_ENV))
effective_batch = int(recipe['effective_batch'])
group_size = int(recipe.get('augmentation_group_size', 2))
preferred_batch = int(recipe['batch_size'])
if PHYSICAL_BATCH_SIZE is None:
    candidates = [preferred_batch, effective_batch // world_size, group_size]
    PHYSICAL_BATCH_SIZE = next(
        (candidate for candidate in candidates if candidate >= group_size and candidate % group_size == 0 and effective_batch % (world_size * candidate) == 0),
        None,
    )
if PHYSICAL_BATCH_SIZE is None or PHYSICAL_BATCH_SIZE % group_size or effective_batch % (world_size * PHYSICAL_BATCH_SIZE):
    raise ValueError(
        f'world_size={world_size}, physical_batch={PHYSICAL_BATCH_SIZE} cannot satisfy the virtual-batch contract '
        f'for effective_batch={effective_batch}, group_size={group_size}'
    )
grad_accum = effective_batch // (world_size * PHYSICAL_BATCH_SIZE)
print({'world_size': world_size, 'physical_batch': PHYSICAL_BATCH_SIZE, 'grad_accum': grad_accum, 'effective_batch': effective_batch, 'group_size': group_size})

output_dir = RUN_ROOT / RUN_NAME
checkpoint = output_dir / 'checkpoint_last.pt'
control_path = CONTROL_ROOT / f'{RUN_NAME}.json'
lock_path = CONTROL_ROOT / f'{RUN_NAME}.lock'
log_path = CONTROL_ROOT / f'{RUN_NAME}-{int(time.time())}.log'
if control_path.is_file():
    existing = json.loads(control_path.read_text())
    try:
        os.kill(int(existing['pid']), 0)
    except OSError:
        control_path.unlink()
    else:
        existing_pid = existing['pid']
        raise RuntimeError(f'a job for {RUN_NAME} is already running as PID {existing_pid}')

command = [
    str(TARGET_PYTHON), '-m', 'torch.distributed.run', '--standalone',
    '--nproc_per_node', str(world_size), 'experiments/train_imagenet.py',
    '--config', str(CONFIG_PATH), '--tier', TIER, '--data-root', str(DATA_ROOT),
    '--output', str(output_dir), '--batch-size', str(PHYSICAL_BATCH_SIZE),
    '--grad-accum', str(grad_accum),
]
for flag, workers in (('--train-workers', TRAIN_WORKERS), ('--val-workers', VAL_WORKERS)):
    if workers is not None:
        if workers < 0:
            raise ValueError(f'{flag} must be non-negative')
        command += [flag, str(workers)]
if checkpoint.is_file():
    command += ['--phase', PHASE, '--resume', str(checkpoint)]
elif output_dir.exists() and any(output_dir.iterdir()):
    raise RuntimeError(f'{output_dir} is nonempty without checkpoint_last.pt; choose a new RUN_NAME')
else:
    command += PHASE_ARGS

launch = ['flock', '-n', str(lock_path), *command]
launch_env = TARGET_ENV.copy()
if CUDA_VISIBLE_DEVICES is not None:
    launch_env['CUDA_VISIBLE_DEVICES'] = CUDA_VISIBLE_DEVICES
show_command(launch)
with log_path.open('ab') as log_handle:
    process = subprocess.Popen(
        launch, cwd=REPOSITORY_DIR, env=launch_env, stdout=log_handle, stderr=subprocess.STDOUT, start_new_session=True
    )
time.sleep(1)
if process.poll() is not None:
    print(log_path.read_text(errors='replace')[-4000:])
    raise RuntimeError(f'launcher exited immediately with code {process.returncode}')
control_path.write_text(json.dumps({'pid': process.pid, 'command': command, 'output': str(output_dir), 'log': str(log_path)}, indent=2) + '\n')
print(f'launched PID {process.pid}; rerun the monitor cell to refresh status')


In [ ]:
from IPython.display import clear_output, display

def tail(path: Path, lines: int = 40) -> str:
    if not path.is_file():
        return '(no log yet)'
    with path.open('rb') as handle:
        handle.seek(max(0, path.stat().st_size - 16384))
        return '\n'.join(handle.read().decode(errors='replace').splitlines()[-lines:])

def refresh_monitor() -> None:
    clear_output(wait=True)
    if not control_path.is_file():
        print('no notebook-controlled job is registered for', RUN_NAME)
        return
    control = json.loads(control_path.read_text())
    pid = int(control['pid'])
    try:
        os.kill(pid, 0)
        state = 'running'
    except OSError:
        state = 'stopped'
    print({'state': state, 'pid': pid, 'output': control['output'], 'log': control['log']})
    metrics_path = Path(control['output']) / 'metrics.jsonl'
    if metrics_path.is_file():
        records = [json.loads(line) for line in metrics_path.read_text().splitlines() if line.strip()]
        if records:
            print('latest metric:', json.dumps(records[-1], indent=2))
    smi = subprocess.run(
        ['nvidia-smi', '--query-gpu=index,name,utilization.gpu,memory.used,memory.total', '--format=csv,noheader'],
        text=True, capture_output=True,
    )
    if smi.returncode == 0:
        print('GPU:', smi.stdout.strip())
    print('\nlast log lines:\n' + tail(Path(control['log'])))

try:
    import ipywidgets as widgets
    refresh_button = widgets.Button(description='Refresh training status', icon='refresh')
    refresh_button.on_click(lambda _: refresh_monitor())
    display(refresh_button)
except ImportError:
    print('ipywidgets is unavailable; rerun this cell to refresh')
refresh_monitor()


In [ ]:
# Set STOP_NOTEBOOK_JOB = True only when intentionally stopping this notebook-managed process group.
STOP_NOTEBOOK_JOB = False
if STOP_NOTEBOOK_JOB:
    if not control_path.is_file():
        raise RuntimeError('no notebook-controlled job is registered')
    import signal
    control = json.loads(control_path.read_text())
    os.killpg(int(control['pid']), signal.SIGTERM)
    print('sent SIGTERM to process group', control['pid'])
